In [2]:
###This script processes each shapefile in a given folder, determines which outflow point (PB or TJ) each feature is closest to, and merges features by nearest outflow. The result is saved as new shapefiles with "_algo_PB" or "_algo_TJ" suffixes. All distance calculations are performed in UTM Zone 11N (EPSG:32611)

import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import MultiPolygon
import os

# Paths
input_dir = "/Volumes/External/TJ/015_shapefiles/autoSAR_output"
outflow_path = "/Volumes/External/TJ/015_shapefiles/outflow_PB_TJ/Outflow.shp"
output_dir = "/Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ"

# UTM Zone 11N (Southern California)
projected_crs = "EPSG:32611"

# Load and project outflow points
print(f"Loading outflow points from: {outflow_path}")
outflow_gdf = gpd.read_file(outflow_path)
outflow_gdf = outflow_gdf.to_crs(projected_crs)
print(f"Loaded {len(outflow_gdf)} outflow points (projected to {projected_crs}).")

# Get list of valid shapefiles to process
shapefiles = [
    f for f in os.listdir(input_dir)
    if f.endswith(".shp") and not f.startswith("._")
]
print(f"Found {len(shapefiles)} shapefiles to process in '{input_dir}'.")

for shp_name in shapefiles:
    input_path = os.path.join(input_dir, shp_name)
    print(f"\nProcessing shapefile: {shp_name}")
    gdf = gpd.read_file(input_path)
    gdf = gdf.to_crs(projected_crs)  # Project to UTM

    print(f"  Number of features in shapefile: {len(gdf)}")

    # Map to store features closest to PB or TJ
    grouped_features = {"PB": [], "TJ": []}

    for i, feature in gdf.iterrows():
        geom = feature.geometry
        distances = outflow_gdf.geometry.distance(geom)
        nearest_idx = distances.idxmin()
        nearest_label = outflow_gdf.loc[nearest_idx, "featureNam"]
        print(f"    Feature {i} is closest to: {nearest_label}")

        if nearest_label in grouped_features:
            grouped_features[nearest_label].append(geom)

    base_name = os.path.splitext(shp_name)[0]

    # Save merged features
    for label, features in grouped_features.items():
        if not features:
            print(f"  No features closest to {label}, skipping.")
            continue
        merged_geom = unary_union(features)
        if not isinstance(merged_geom, MultiPolygon):
            merged_geom = MultiPolygon([merged_geom])

        merged_gdf = gpd.GeoDataFrame({"geometry": [merged_geom]}, crs=projected_crs)
        output_name = f"{base_name}_algo_{label}.shp"
        output_path = os.path.join(output_dir, output_name)
        merged_gdf.to_file(output_path)
        print(f"  Saved merged {label} geometry to: {output_path}")

print("\nAll shapefiles processed successfully.")

Loading outflow points from: /Volumes/External/TJ/015_shapefiles/outflow_PB_TJ/Outflow.shp
Loaded 2 outflow points (projected to EPSG:32611).
Found 205 shapefiles to process in '/Volumes/External/TJ/015_shapefiles/autoSAR_output'.

Processing shapefile: S1A_IW_GRDH_1SDV_20220101T015007_20220101T015036_041261_04E75E_B606_pre_reordered_mask.shp
  Number of features in shapefile: 7
    Feature 0 is closest to: TJ
    Feature 1 is closest to: PB
    Feature 2 is closest to: PB
    Feature 3 is closest to: PB
    Feature 4 is closest to: PB
    Feature 5 is closest to: PB
    Feature 6 is closest to: PB
  Saved merged PB geometry to: /Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/S1A_IW_GRDH_1SDV_20220101T015007_20220101T015036_041261_04E75E_B606_pre_reordered_mask_algo_PB.shp
  Saved merged TJ geometry to: /Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/S1A_IW_GRDH_1SDV_20220101T015007_20220101T015036_041261_04E75E_B606_pre_reordered_mask_algo_TJ.shp

Processing shapefil

In [7]:
# ============================================================
# Compute and plot IoU between observed plume polygons and
# algorithm output shapefiles.
#
# Matching logic:
#   Observed sourceName:
#       S1A_IW_GRDH_1SDV_20221203T015015_20221203T015044_046161_0586BC_72DC
#
#   Predicted shapefile:
#       S1A_IW_GRDH_1SDV_20221203T015015_20221203T015044_046161_0586BC_72DC_pre_reordered_mask_algo_PB.shp
#
#   Match by:
#       1. full Sentinel start timestamp, e.g. 20221203T015015
#       2. outflow label, e.g. PB or TJ
#
# Background TIFF:
#   sourceName + "_pre_reordered.tif"
#
# Outputs:
#   1. iou_summary.csv
#   2. One PNG plot per observed-predicted comparison
#
# All IoU area calculations are done in UTM Zone 11N, EPSG:32611.
# ============================================================

import os
import glob
import re
import numpy as np
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

from rasterio.plot import plotting_extent
from rasterio.warp import calculate_default_transform, reproject, Resampling
from shapely.ops import unary_union
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


# ----------------------------
# Paths
# ----------------------------

observed_path = "/Volumes/External/TJ/015_shapefiles/shapefiles/plumeExtentv4.shp"

predicted_dir = "/Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ"

outflow_path = "/Volumes/External/TJ/015_shapefiles/outflow_PB_TJ/Outflow.shp"

background_tif_dir = "/Volumes/External/TJ/020_preprocessing/sentinel1/reordered_for_autoSAR"

output_csv = "/Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/iou_summary.csv"

output_plot_dir = "/Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/iou_plots"


# UTM Zone 11N, Southern California
projected_crs = "EPSG:32611"

os.makedirs(output_plot_dir, exist_ok=True)


# ============================================================
# Helper functions
# ============================================================

def extract_sentinel_start_time(text):
    """
    Extracts the first Sentinel timestamp from a filename or sourceName.

    Example:
        S1A_IW_GRDH_1SDV_20221203T015015_20221203T015044_...
    returns:
        20221203T015015
    """

    if text is None:
        return None

    text = str(text)

    match = re.search(r"\d{8}T\d{6}", text)

    if match:
        return match.group(0)

    return None


def extract_sentinel_date(text):
    """
    Extracts YYYYMMDD from a Sentinel filename or sourceName.
    """

    start_time = extract_sentinel_start_time(text)

    if start_time is None:
        return None

    return start_time[:8]


def find_background_tif(source_name, background_tif_dir):
    """
    Finds original reordered SAR TIFF using:
        sourceName + "_pre_reordered.tif"
    """

    tif_name = f"{source_name}_pre_reordered.tif"
    tif_path = os.path.join(background_tif_dir, tif_name)

    if os.path.exists(tif_path):
        return tif_path

    print(f"  Background TIFF not found: {tif_name}")
    return None


def find_predicted_shapefile(source_name, outflow, predicted_dir):
    """
    Finds predicted shapefile by matching:
        1. full Sentinel start timestamp, e.g. 20221203T015015
        2. outflow suffix, e.g. _algo_PB.shp or _algo_TJ.shp
    """

    source_start_time = extract_sentinel_start_time(source_name)

    if source_start_time is None:
        print(f"  Could not extract start time from sourceName: {source_name}")
        return None

    search_pattern = os.path.join(
        predicted_dir,
        f"*{source_start_time}*_algo_{outflow}.shp"
    )

    matches = sorted(glob.glob(search_pattern))

    if len(matches) == 0:
        print(
            f"  No predicted shapefile found for start time "
            f"{source_start_time} and outflow {outflow}"
        )
        return None

    if len(matches) > 1:
        print(
            f"  WARNING: Multiple predicted shapefiles found for "
            f"start time {source_start_time} and outflow {outflow}"
        )
        for match_path in matches:
            print(f"    {os.path.basename(match_path)}")
        print("  Using the first match.")

    return matches[0]


def fix_geometry(geom):
    """
    Attempts to fix invalid geometries.
    """

    if geom is None:
        return None

    if geom.is_empty:
        return None

    if not geom.is_valid:
        geom = geom.buffer(0)

    if geom.is_empty:
        return None

    return geom


def merge_valid_geometries(gdf):
    """
    Fixes and merges all valid geometries in a GeoDataFrame.
    """

    valid_geoms = []

    for geom in gdf.geometry:
        fixed_geom = fix_geometry(geom)

        if fixed_geom is not None:
            valid_geoms.append(fixed_geom)

    if len(valid_geoms) == 0:
        return None

    return unary_union(valid_geoms)


def calculate_iou(obs_geom, pred_geom):
    """
    Calculates IoU and returns area details.
    """

    obs_geom = fix_geometry(obs_geom)
    pred_geom = fix_geometry(pred_geom)

    if obs_geom is None or pred_geom is None:
        return {
            "observed_area_m2": 0,
            "predicted_area_m2": 0,
            "intersection_area_m2": 0,
            "union_area_m2": 0,
            "iou": 0,
            "intersection_geom": None
        }

    intersection_geom = obs_geom.intersection(pred_geom)

    observed_area = obs_geom.area
    predicted_area = pred_geom.area
    intersection_area = intersection_geom.area
    union_area = obs_geom.union(pred_geom).area

    if union_area > 0:
        iou = intersection_area / union_area
    else:
        iou = 0

    return {
        "observed_area_m2": observed_area,
        "predicted_area_m2": predicted_area,
        "intersection_area_m2": intersection_area,
        "union_area_m2": union_area,
        "iou": iou,
        "intersection_geom": intersection_geom
    }


def safe_filename(text):
    """
    Makes a string safe to use in output filenames.
    """

    text = str(text)
    text = re.sub(r"[^A-Za-z0-9_\-]+", "_", text)

    return text[:180]


def plot_background_tif(ax, tif_path, target_crs):
    """
    Adds SAR TIFF as a grayscale background.

    If the TIFF CRS differs from target_crs, it reprojects the raster
    in memory before plotting so it aligns with the shapefiles.

    This does not permanently modify the TIFF on disk.
    """

    if tif_path is None:
        return None

    with rasterio.open(tif_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

        if raster_crs is None:
            print(
                "  WARNING: Background TIFF has no CRS. "
                "Cannot safely align with shapefiles."
            )
            return None

        # ----------------------------------------------------
        # Case 1: TIFF is already in target CRS
        # ----------------------------------------------------
        if str(raster_crs) == target_crs:
            arr = src.read(1).astype(np.float32)
            extent = plotting_extent(src)

        # ----------------------------------------------------
        # Case 2: Reproject TIFF in memory to target CRS
        # ----------------------------------------------------
        else:
            print(
                f"  Reprojecting background TIFF from {raster_crs} "
                f"to {target_crs} for plotting."
            )

            transform, width, height = calculate_default_transform(
                src.crs,
                target_crs,
                src.width,
                src.height,
                *src.bounds
            )

            arr = np.empty((height, width), dtype=np.float32)

            reproject(
                source=rasterio.band(src, 1),
                destination=arr,
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=nodata,
                dst_transform=transform,
                dst_crs=target_crs,
                dst_nodata=np.nan,
                resampling=Resampling.nearest
            )

            left = transform.c
            top = transform.f
            right = left + transform.a * width
            bottom = top + transform.e * height

            extent = (left, right, bottom, top)

        # ----------------------------------------------------
        # Mask NoData
        # ----------------------------------------------------
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)

        valid = arr[np.isfinite(arr)]

        if valid.size == 0:
            print("  Background TIFF has no valid pixels.")
            return raster_crs

        # Robust grayscale stretch
        vmin, vmax = np.nanpercentile(valid, [2, 98])

        ax.imshow(
            arr,
            extent=extent,
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
            origin="upper"
        )

        return target_crs


def plot_iou(
    obs_geom,
    pred_geom,
    intersection_geom,
    outflow_gdf,
    source_name,
    source_start_time,
    source_date,
    outflow,
    pred_name,
    iou,
    output_plot_dir,
    observed_index,
    background_tif_dir
):
    """
    Plots SAR background, observed geometry, predicted geometry,
    overlap, and outflow point.
    """

    fig, ax = plt.subplots(figsize=(9, 9))

    # --------------------------------------------------------
    # Background TIFF
    # --------------------------------------------------------

    background_tif = find_background_tif(
        source_name=source_name,
        background_tif_dir=background_tif_dir
    )

    plot_background_tif(
        ax=ax,
        tif_path=background_tif,
        target_crs=projected_crs
    )

    # --------------------------------------------------------
    # Shapefile geometries
    # --------------------------------------------------------

    obs_plot_gdf = gpd.GeoDataFrame(
        {"geometry": [obs_geom]},
        crs=projected_crs
    )

    pred_plot_gdf = gpd.GeoDataFrame(
        {"geometry": [pred_geom]},
        crs=projected_crs
    )

    # Transparent fills
    obs_plot_gdf.plot(
        ax=ax,
        color="cyan",
        alpha=0.00
    )

    pred_plot_gdf.plot(
        ax=ax,
        color="magenta",
        alpha=0.00
    )

    # Strong outlines
    obs_plot_gdf.plot(
        ax=ax,
        facecolor="none",
        edgecolor="cyan",
        linewidth=2.5
    )

    pred_plot_gdf.plot(
        ax=ax,
        facecolor="none",
        edgecolor="magenta",
        linewidth=2.5
    )

    # Intersection / overlap
    if intersection_geom is not None and not intersection_geom.is_empty:
        intersection_plot_gdf = gpd.GeoDataFrame(
            {"geometry": [intersection_geom]},
            crs=projected_crs
        )

        intersection_plot_gdf.plot(
            ax=ax,
            color="yellow",
            alpha=0.65
        )

    # Outflow point
    if outflow_gdf is not None and "featureNam" in outflow_gdf.columns:
        relevant_outflow = outflow_gdf[outflow_gdf["featureNam"] == outflow]

        if len(relevant_outflow) > 0:
            relevant_outflow.plot(
                ax=ax,
                marker="*",
                color="black",
                markersize=150
            )

    # --------------------------------------------------------
    # Title, extent, legend
    # --------------------------------------------------------

    if background_tif is not None:
        background_name = os.path.basename(background_tif)
    else:
        background_name = "No background TIFF found"

    ax.set_title(
        f"{source_start_time} | {outflow} | IoU = {iou:.4f}\n"
        f"Background: {background_name}\n"
        f"Prediction: {pred_name}",
        fontsize=10
    )

    ax.set_xlabel("Easting, UTM Zone 11N (m)")
    ax.set_ylabel("Northing, UTM Zone 11N (m)")
    ax.set_aspect("equal")

    # Set plot extent around observed + predicted union
    bounds = obs_geom.union(pred_geom).bounds
    minx, miny, maxx, maxy = bounds

    pad_x = (maxx - minx) * 0.25
    pad_y = (maxy - miny) * 0.25

    if pad_x == 0:
        pad_x = 100

    if pad_y == 0:
        pad_y = 100

    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    legend_elements = [
        Patch(
            facecolor="cyan",
            edgecolor="cyan",
            alpha=0.25,
            label="Observed plume"
        ),
        Patch(
            facecolor="magenta",
            edgecolor="magenta",
            alpha=0.25,
            label="Predicted plume"
        ),
        Patch(
            facecolor="yellow",
            edgecolor="yellow",
            alpha=0.65,
            label="Intersection"
        ),
        Line2D(
            [0],
            [0],
            marker="*",
            color="black",
            label=f"{outflow} outflow",
            markerfacecolor="black",
            markersize=12,
            linestyle="None"
        )
    ]

    ax.legend(handles=legend_elements, loc="best")

    plt.tight_layout()

    plot_name = (
        f"{observed_index:03d}_"
        f"{source_start_time}_"
        f"{outflow}_"
        f"IoU_{iou:.4f}_"
        f"{safe_filename(source_name)}.png"
    )

    plot_path = os.path.join(output_plot_dir, plot_name)

    plt.savefig(plot_path, dpi=300)
    plt.close()

    return plot_path, background_tif


# ============================================================
# Load observed plume shapefile
# ============================================================

print(f"Loading observed shapefile: {observed_path}")

obs_gdf = gpd.read_file(observed_path)
obs_gdf = obs_gdf.to_crs(projected_crs)

print(f"Loaded {len(obs_gdf)} observed plume features.")
print("Observed shapefile columns:")
print(list(obs_gdf.columns))


# ============================================================
# Load outflow points
# ============================================================

if os.path.exists(outflow_path):
    outflow_gdf = gpd.read_file(outflow_path)
    outflow_gdf = outflow_gdf.to_crs(projected_crs)
    print(f"Loaded {len(outflow_gdf)} outflow points.")
else:
    outflow_gdf = None
    print("Outflow shapefile not found. Plots will not include outflow point.")


# ============================================================
# Check required observed shapefile fields
# ============================================================

required_columns = ["sourceName", "outflow"]

for col in required_columns:
    if col not in obs_gdf.columns:
        raise ValueError(
            f"Required column '{col}' was not found in the observed shapefile."
        )


# ============================================================
# Compute and plot IoU
# ============================================================

results = []

for idx, row in obs_gdf.iterrows():

    source_name = row["sourceName"]
    outflow = row["outflow"]

    if outflow is None:
        print(f"\nObserved feature {idx} has no outflow value. Skipping.")
        continue

    outflow = str(outflow).strip()

    obs_geom = fix_geometry(row.geometry)

    source_start_time = extract_sentinel_start_time(source_name)

    if source_start_time is not None:
        source_date = source_start_time[:8]
    else:
        source_date = None

    print("\n--------------------------------------------------")
    print(f"Observed feature index: {idx}")
    print(f"  sourceName: {source_name}")
    print(f"  outflow: {outflow}")
    print(f"  extracted start time: {source_start_time}")
    print(f"  extracted date: {source_date}")

    if obs_geom is None:
        print("  Observed geometry is empty or invalid. IoU set to 0.")

        results.append({
            "observed_index": idx,
            "sourceName": source_name,
            "source_start_time": source_start_time,
            "source_date": source_date,
            "outflow": outflow,
            "predicted_file": None,
            "predicted_found": False,
            "background_tif": None,
            "plot_path": None,
            "observed_area_m2": 0,
            "predicted_area_m2": 0,
            "intersection_area_m2": 0,
            "union_area_m2": 0,
            "iou": 0
        })

        continue

    pred_path = find_predicted_shapefile(
        source_name=source_name,
        outflow=outflow,
        predicted_dir=predicted_dir
    )

    if pred_path is None:
        print("  Predicted shapefile not found. IoU set to 0.")

        background_tif = find_background_tif(
            source_name=source_name,
            background_tif_dir=background_tif_dir
        )

        results.append({
            "observed_index": idx,
            "sourceName": source_name,
            "source_start_time": source_start_time,
            "source_date": source_date,
            "outflow": outflow,
            "predicted_file": None,
            "predicted_found": False,
            "background_tif": background_tif,
            "plot_path": None,
            "observed_area_m2": obs_geom.area,
            "predicted_area_m2": 0,
            "intersection_area_m2": 0,
            "union_area_m2": obs_geom.area,
            "iou": 0
        })

        continue

    pred_name = os.path.basename(pred_path)

    print(f"  matched predicted file: {pred_name}")

    pred_gdf = gpd.read_file(pred_path)
    pred_gdf = pred_gdf.to_crs(projected_crs)

    pred_geom = merge_valid_geometries(pred_gdf)

    if pred_geom is None:
        print("  Predicted shapefile has no valid geometries. IoU set to 0.")

        background_tif = find_background_tif(
            source_name=source_name,
            background_tif_dir=background_tif_dir
        )

        results.append({
            "observed_index": idx,
            "sourceName": source_name,
            "source_start_time": source_start_time,
            "source_date": source_date,
            "outflow": outflow,
            "predicted_file": pred_name,
            "predicted_found": True,
            "background_tif": background_tif,
            "plot_path": None,
            "observed_area_m2": obs_geom.area,
            "predicted_area_m2": 0,
            "intersection_area_m2": 0,
            "union_area_m2": obs_geom.area,
            "iou": 0
        })

        continue

    iou_stats = calculate_iou(obs_geom, pred_geom)

    print(f"  observed area:     {iou_stats['observed_area_m2']:.2f} m²")
    print(f"  predicted area:    {iou_stats['predicted_area_m2']:.2f} m²")
    print(f"  intersection area: {iou_stats['intersection_area_m2']:.2f} m²")
    print(f"  union area:        {iou_stats['union_area_m2']:.2f} m²")
    print(f"  IoU:               {iou_stats['iou']:.4f}")

    plot_path, background_tif = plot_iou(
        obs_geom=obs_geom,
        pred_geom=pred_geom,
        intersection_geom=iou_stats["intersection_geom"],
        outflow_gdf=outflow_gdf,
        source_name=source_name,
        source_start_time=source_start_time,
        source_date=source_date,
        outflow=outflow,
        pred_name=pred_name,
        iou=iou_stats["iou"],
        output_plot_dir=output_plot_dir,
        observed_index=idx,
        background_tif_dir=background_tif_dir
    )

    print(f"  background TIFF: {background_tif}")
    print(f"  saved IoU plot: {plot_path}")

    results.append({
        "observed_index": idx,
        "sourceName": source_name,
        "source_start_time": source_start_time,
        "source_date": source_date,
        "outflow": outflow,
        "predicted_file": pred_name,
        "predicted_found": True,
        "background_tif": background_tif,
        "plot_path": plot_path,
        "observed_area_m2": iou_stats["observed_area_m2"],
        "predicted_area_m2": iou_stats["predicted_area_m2"],
        "intersection_area_m2": iou_stats["intersection_area_m2"],
        "union_area_m2": iou_stats["union_area_m2"],
        "iou": iou_stats["iou"]
    })


# ============================================================
# Save results
# ============================================================

iou_df = pd.DataFrame(results)

os.makedirs(os.path.dirname(output_csv), exist_ok=True)

iou_df.to_csv(output_csv, index=False)

print("\n==================================================")
print("IoU calculation and plotting complete.")
print(f"Saved summary CSV to: {output_csv}")
print(f"Saved IoU plots to: {output_plot_dir}")
print("==================================================")

iou_df

Loading observed shapefile: /Volumes/External/TJ/015_shapefiles/shapefiles/plumeExtentv4.shp
Loaded 64 observed plume features.
Observed shapefile columns:
['fid_1', 'sourceName', 'fileName', 'outflow', 'Shape_Area', 'area_SQKM', 'geometry']
Loaded 2 outflow points.

--------------------------------------------------
Observed feature index: 0
  sourceName: S1A_IW_GRDH_1SDV_20241216T015014_20241216T015043_057011_070161_E8F2
  outflow: PB
  extracted start time: 20241216T015014
  extracted date: 20241216
  matched predicted file: S1A_IW_GRDH_1SDV_20241216T015014_20241216T015043_057011_070161_E8F2_pre_reordered_mask_algo_PB.shp
  observed area:     2219405.58 m²
  predicted area:    1247630.04 m²
  intersection area: 985897.32 m²
  union area:        2481138.30 m²
  IoU:               0.3974
  Reprojecting background TIFF from EPSG:4326 to EPSG:32611 for plotting.
  background TIFF: /Volumes/External/TJ/020_preprocessing/sentinel1/reordered_for_autoSAR/S1A_IW_GRDH_1SDV_20241216T015014_202

/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:131: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)
/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)
/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


  background TIFF: /Volumes/External/TJ/020_preprocessing/sentinel1/reordered_for_autoSAR/S1A_IW_GRDH_1SDV_20240923T015016_20240923T015045_055786_06D0CD_4C0A_pre_reordered.tif
  saved IoU plot: /Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/iou_plots/002_20240923T015016_PB_IoU_0.7325_S1A_IW_GRDH_1SDV_20240923T015016_20240923T015045_055786_06D0CD_4C0A.png

--------------------------------------------------
Observed feature index: 3
  sourceName: S1A_IW_GRDH_1SDV_20240830T015015_20240830T015045_055436_06C312_73FC
  outflow: PB
  extracted start time: 20240830T015015
  extracted date: 20240830
  matched predicted file: S1A_IW_GRDH_1SDV_20240830T015015_20240830T015045_055436_06C312_73FC_pre_reordered_mask_algo_PB.shp
  observed area:     2125156.78 m²
  predicted area:    1561873.34 m²
  intersection area: 1465569.97 m²
  union area:        2221460.15 m²
  IoU:               0.6597
  Reprojecting background TIFF from EPSG:4326 to EPSG:32611 for plotting.


/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:131: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)
/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)
/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


  background TIFF: /Volumes/External/TJ/020_preprocessing/sentinel1/reordered_for_autoSAR/S1A_IW_GRDH_1SDV_20240830T015015_20240830T015045_055436_06C312_73FC_pre_reordered.tif
  saved IoU plot: /Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/iou_plots/003_20240830T015015_PB_IoU_0.6597_S1A_IW_GRDH_1SDV_20240830T015015_20240830T015045_055436_06C312_73FC.png

--------------------------------------------------
Observed feature index: 4
  sourceName: S1A_IW_GRDH_1SDV_20240923T015016_20240923T015045_055786_06D0CD_4C0A
  outflow: TJ
  extracted start time: 20240923T015016
  extracted date: 20240923
  matched predicted file: S1A_IW_GRDH_1SDV_20240923T015016_20240923T015045_055786_06D0CD_4C0A_pre_reordered_mask_algo_TJ.shp
  observed area:     341941.95 m²
  predicted area:    275475.25 m²
  intersection area: 251338.05 m²
  union area:        366079.15 m²
  IoU:               0.6866
  Reprojecting background TIFF from EPSG:4326 to EPSG:32611 for plotting.


...

/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:131: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)
/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)
/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


  background TIFF: /Volumes/External/TJ/020_preprocessing/sentinel1/reordered_for_autoSAR/S1A_IW_GRDH_1SDV_20250202T135312_20250202T135337_057718_071D69_4DDB_pre_reordered.tif
  saved IoU plot: /Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/iou_plots/062_20250202T135312_PB_IoU_0.0911_S1A_IW_GRDH_1SDV_20250202T135312_20250202T135337_057718_071D69_4DDB.png

--------------------------------------------------
Observed feature index: 63
  sourceName: S1A_IW_GRDH_1SDV_20240911T015016_20240911T015045_055611_06C9E7_3B08
  outflow: PB
  extracted start time: 20240911T015016
  extracted date: 20240911
  matched predicted file: S1A_IW_GRDH_1SDV_20240911T015016_20240911T015045_055611_06C9E7_3B08_pre_reordered_mask_algo_PB.shp
  observed area:     2651661.39 m²
  predicted area:    3867074.19 m²
  intersection area: 1970446.50 m²
  union area:        4548289.08 m²
  IoU:               0.4332
  Reprojecting background TIFF from EPSG:4326 to EPSG:32611 for plotting.


/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:131: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)
/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)
/opt/anaconda3/envs/RS/lib/python3.9/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


  background TIFF: /Volumes/External/TJ/020_preprocessing/sentinel1/reordered_for_autoSAR/S1A_IW_GRDH_1SDV_20240911T015016_20240911T015045_055611_06C9E7_3B08_pre_reordered.tif
  saved IoU plot: /Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/iou_plots/063_20240911T015016_PB_IoU_0.4332_S1A_IW_GRDH_1SDV_20240911T015016_20240911T015045_055611_06C9E7_3B08.png

IoU calculation and plotting complete.
Saved summary CSV to: /Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/iou_summary.csv
Saved IoU plots to: /Volumes/External/TJ/015_shapefiles/autoSAR_output_PB_TJ/iou_plots


,observed_index,sourceName,source_start_time,source_date,outflow,predicted_file,predicted_found,background_tif,plot_path,observed_area_m2,predicted_area_m2,intersection_area_m2,union_area_m2,iou
0,0,S1A_IW_GRDH_1SDV_20241216T015014_20241216T0150...,20241216T015014,20241216,PB,S1A_IW_GRDH_1SDV_20241216T015014_20241216T0150...,True,/Volumes/External/TJ/020_preprocessing/sentine...,/Volumes/External/TJ/015_shapefiles/autoSAR_ou...,2.219406e+06,1.247630e+06,9.858973e+05,2.481138e+06,0.397357
1,1,S1A_IW_GRDH_1SDV_20240930T134456_20240930T1345...,20240930T134456,20240930,PB,S1A_IW_GRDH_1SDV_20240930T134456_20240930T1345...,True,/Volumes/External/TJ/020_preprocessing/sentine...,/Volumes/External/TJ/015_shapefiles/autoSAR_ou...,7.073144e+06,7.636368e+04,7.636368e+04,7.073144e+06,0.010796
2,2,S1A_IW_GRDH_1SDV_20240923T015016_20240923T0150...,20240923T015016,20240923,PB,S1A_IW_GRDH_1SDV_20240923T015016_20240923T0150...,True,/Volumes/External/TJ/020_preprocessing/sentine...,/Volumes/External/TJ/015_shapefiles/autoSAR_ou...,4.725812e+06,5.208073e+06,4.200046e+06,5.733839e+06,0.732502
3,3,S1A_IW_GRDH_1SDV_20240830T015015_20240830T0150...,20240830T015015,20240830,PB,S1A_IW_GRDH_1SDV_20240830T015015_20240830T0150...,True,/Volumes/External/TJ/020_preprocessing/sentine...,/Volumes/External/TJ/015_shapefiles/autoSAR_ou...,2.125157e+06,1.561873e+06,1.465570e+06,2.221460e+06,0.659733
4,4,S1A_IW_GRDH_1SDV_20240923T015016_20240923T0150...,20240923T015016,20240923,TJ,S1A_IW_GRDH_1SDV_20240923T015016_20240923T0150...,True,/Volumes/External/TJ/020_preprocessing/sentine...,/Volumes/External/TJ/015_shapefiles/autoSAR_ou...,3.419420e+05,2.754752e+05,2.513380e+05,3.660792e+05,0.686567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,59,S1A_IW_GRDH_1SDV_20240327T135319_20240327T1353...,20240327T135319,20240327,TJ,None,False,/Volumes/External/TJ/020_preprocessing/sentine...,None,4.194446e+06,0.000000e+00,0.000000e+00,4.194446e+06,0.000000
60,60,S1A_IW_GRDH_1SDV_20240327T135319_20240327T1353...,20240327T135319,20240327,PB,S1A_IW_GRDH_1SDV_20240327T135319_20240327T1353...,True,/Volumes/External/TJ/020_preprocessing/sentine...,/Volumes/External/TJ/015_shapefiles/autoSAR_ou...,1.597598e+07,1.577987e+05,1.577987e+05,1.597598e+07,0.009877
61,61,S1A_IW_GRDH_1SDV_20250202T135312_20250202T1353...,20250202T135312,20250202,TJ,S1A_IW_GRDH_1SDV_20250202T135312_20250202T1353...,True,/Volumes/External/TJ/020_preprocessing/sentine...,/Volumes/External/TJ/015_shapefiles/autoSAR_ou...,2.815440e+07,1.492768e+08,2.531950e+07,1.521117e+08,0.166453
62,62,S1A_IW_GRDH_1SDV_20250202T135312_20250202T1353...,20250202T135312,20250202,PB,S1A_IW_GRDH_1SDV_20250202T135312_20250202T1353...,True,/Volumes/External/TJ/020_preprocessing/sentine...,/Volumes/External/TJ/015_shapefiles/autoSAR_ou...,2.276691e+07,2.835013e+06,2.136583e+06,2.346534e+07,0.091053
